# 02 — Pricing American Options

American options can be exercised at any time up to maturity. This notebook
shows how to price them with:

1. **Binomial tree** (CRR with backward induction)
2. **PDE finite difference** (Crank-Nicolson with PSOR early-exercise)
3. **Monte Carlo** (Longstaff-Schwartz regression)

We also demonstrate the **early exercise premium** and the
**control variate** adjustment.

## 1) Setup

In [1]:
import datetime as dt

import derivatives_pricing as dp

## 2) Market Data & Contract

In [2]:
pricing_date = dt.datetime(2025, 6, 15)
maturity = dt.datetime(2026, 6, 15)
r = 0.05

curve = dp.DiscountCurve.flat(rate=r, end_time=2.0)
market = dp.MarketData(pricing_date=pricing_date, discount_curve=curve, currency="USD")

underlying = dp.UnderlyingData(initial_value=100.0, volatility=0.20, market_data=market)

am_put = dp.VanillaSpec(
    option_type=dp.OptionType.PUT,
    exercise_type=dp.ExerciseType.AMERICAN,
    strike=105.0,
    maturity=maturity,
)

## 3) Binomial Tree

The CRR binomial tree naturally handles early exercise via backward induction.

In [3]:
binom = dp.OptionValuation(
    underlying=underlying,
    spec=am_put,
    pricing_method=dp.PricingMethod.BINOMIAL,
    params=dp.BinomialParams(num_steps=500),
)
print(f"Binomial American put: {binom.present_value():.4f}")

Binomial American put: 8.7416


## 4) PDE Finite Difference

The PDE solver uses a projected SOR (PSOR) algorithm to enforce the
early-exercise constraint at each time step.

In [4]:
pde = dp.OptionValuation(
    underlying=underlying,
    spec=am_put,
    pricing_method=dp.PricingMethod.PDE_FD,
    params=dp.PDEParams(),
)
print(f"PDE American put:      {pde.present_value():.4f}")

PDE American put:      8.7414


## 5) Monte Carlo (Longstaff-Schwartz)

MC American pricing uses the Longstaff-Schwartz (2001) least-squares
regression to approximate the continuation value at each exercise date.

In [5]:
sim_config = dp.SimulationConfig(paths=100_000, end_date=maturity, num_steps=100)
gbm = dp.GBMProcess(
    market_data=market,
    process_params=dp.GBMParams(initial_value=100.0, volatility=0.20),
    sim_config=sim_config,
)

mc = dp.OptionValuation(
    underlying=gbm,
    spec=am_put,
    pricing_method=dp.PricingMethod.MONTE_CARLO,
    params=dp.MonteCarloParams(random_seed=42),
)
print(f"MC American put:       {mc.present_value():.4f}")

MC American put:       8.7145


## 6) Early Exercise Premium

The American price exceeds the European price by the early exercise premium.

In [6]:
eu_put = dp.VanillaSpec(
    option_type=dp.OptionType.PUT,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=105.0,
    maturity=maturity,
)

bsm_eu = dp.OptionValuation(underlying=underlying, spec=eu_put, pricing_method=dp.PricingMethod.BSM)

eu_price = bsm_eu.present_value()
am_price = binom.present_value()
print(f"European put: {eu_price:.4f}")
print(f"American put: {am_price:.4f}")
print(f"Early exercise premium: {am_price - eu_price:.4f}")

European put: 7.9004
American put: 8.7416
Early exercise premium: 0.8412


## 7) Control Variate Adjustment

The control variate uses the known BSM European price to reduce bias in
American pricing. It computes:

$$\hat{V}_{AM} = V_{AM}^{num} + (V_{EU}^{BSM} - V_{EU}^{num})$$

This corrects for systematic grid/simulation bias. To see the effect clearly,
we use a **coarse** grid (30 steps).

In [7]:
# Coarse binomial — 30 steps shows noticeable bias vs BSM
binom_coarse = dp.OptionValuation(
    underlying=underlying,
    spec=am_put,
    pricing_method=dp.PricingMethod.BINOMIAL,
    params=dp.BinomialParams(num_steps=30),
)
binom_coarse_cv = dp.OptionValuation(
    underlying=underlying,
    spec=am_put,
    pricing_method=dp.PricingMethod.BINOMIAL,
    params=dp.BinomialParams(num_steps=30, control_variate_european=True),
)

print("Binomial (30 steps):")
print(f"  No CV:   {binom_coarse.present_value():.4f}")
print(f"  With CV: {binom_coarse_cv.present_value():.4f}")
print(f"  Reference (500 steps): {binom.present_value():.4f}")

# Coarse PDE — 30×30 grid
pde_coarse = dp.OptionValuation(
    underlying=underlying,
    spec=am_put,
    pricing_method=dp.PricingMethod.PDE_FD,
    params=dp.PDEParams(spot_steps=30, time_steps=30),
)
pde_coarse_cv = dp.OptionValuation(
    underlying=underlying,
    spec=am_put,
    pricing_method=dp.PricingMethod.PDE_FD,
    params=dp.PDEParams(spot_steps=30, time_steps=30, control_variate_european=True),
)

pde_ref = dp.OptionValuation(
    underlying=underlying,
    spec=am_put,
    pricing_method=dp.PricingMethod.PDE_FD,
    params=dp.PDEParams(),
)

print("\nPDE (30×30 grid):")
print(f"  No CV:   {pde_coarse.present_value():.4f}")
print(f"  With CV: {pde_coarse_cv.present_value():.4f}")
print(f"  Reference (default grid): {pde_ref.present_value():.4f}")

Binomial (30 steps):
  No CV:   8.7667
  With CV: 8.7238
  Reference (500 steps): 8.7416

PDE (30×30 grid):
  No CV:   8.8530
  With CV: 8.6423
  Reference (default grid): 8.7414


## 8) American Call

An American call on a non-dividend-paying stock has no early exercise premium
(it equals the European call). Adding dividends makes early exercise
potentially optimal.

In [8]:
am_call = dp.VanillaSpec(
    option_type=dp.OptionType.CALL,
    exercise_type=dp.ExerciseType.AMERICAN,
    strike=105.0,
    maturity=maturity,
)

# No dividends — American call = European call
binom_am_call = dp.OptionValuation(
    underlying=underlying,
    spec=am_call,
    pricing_method=dp.PricingMethod.BINOMIAL,
    params=dp.BinomialParams(num_steps=500),
)
eu_call = dp.VanillaSpec(
    option_type=dp.OptionType.CALL,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=105.0,
    maturity=maturity,
)
bsm_eu_call = dp.OptionValuation(
    underlying=underlying, spec=eu_call, pricing_method=dp.PricingMethod.BSM
)
print(f"European call (no divs): {bsm_eu_call.present_value():.4f}")
print(f"American call (no divs): {binom_am_call.present_value():.4f}")

# With dividends — early exercise becomes valuable
dividends = [
    (dt.datetime(2025, 9, 15, 0, 0), 3),
    (dt.datetime(2025, 12, 15, 0, 0), 3),
    (dt.datetime(2026, 3, 15, 0, 0), 3),
    (dt.datetime(2026, 6, 15, 0, 0), 3),
]

underlying_div = dp.UnderlyingData(
    initial_value=100.0, volatility=0.20, market_data=market, discrete_dividends=dividends
)

bsm_eu_div = dp.OptionValuation(
    underlying=underlying_div, spec=eu_call, pricing_method=dp.PricingMethod.BSM
)
binom_am_div = dp.OptionValuation(
    underlying=underlying_div,
    spec=am_call,
    pricing_method=dp.PricingMethod.BINOMIAL,
    params=dp.BinomialParams(num_steps=500),
)
print(f"\nEuropean call (with divs): {bsm_eu_div.present_value():.4f}")
print(f"American call (with divs): {binom_am_div.present_value():.4f}")
print(f"Early exercise premium:    {binom_am_div.present_value() - bsm_eu_div.present_value():.4f}")

European call (no divs): 8.0214
American call (no divs): 8.0232

European call (with divs): 3.0983
American call (with divs): 4.1331
Early exercise premium:    1.0348


## 9) Method Comparison

In [9]:
print(f"{'Method':<25} {'American Put':>13}")
print("-" * 40)
for label, val in [
    ("Binomial (500 steps)", binom),
    ("PDE (Crank-Nicolson)", pde),
    ("MC (Longstaff-Schwartz)", mc),
]:
    print(f"{label:<25} {val.present_value():>13.4f}")

Method                     American Put
----------------------------------------
Binomial (500 steps)             8.7416
PDE (Crank-Nicolson)             8.7414
MC (Longstaff-Schwartz)          8.7145
